# 30 — Identifiability probes, Monte-Carlo (Axis 1)

Rewritten to the high-level `irtree_fit` API, run under **both links** over K datasets. Identifiability is a property of the model, so `κ` (identified through b's prior) and `β`/`θ` should behave the same way under probit and logit.

## Setup

In [ ]:
import sys
!{sys.executable} -m pip install -e "C:\\Users\\manonfi\\OneDrive - James Madison University\\Spring 26 classes & projects\\Practicum Python IRtree package\\PACKAGE\\Irtree"

In [ ]:
import sys, warnings, pathlib, importlib.util
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
warnings.filterwarnings('ignore')

import irtree.tree_spec   as ts
import irtree.node_models as nm
import irtree.priors      as pr
import irtree.constraints as con
import irtree.ingest      as ing
import irtree.fit         as ft

# The data-generating code (simulate_irtree) lives in the test harness, NOT in
# the package: simulation is deliberately kept separate from fitting.  Loaded
# the same way notebooks 40 & 50 do.
ROOT = pathlib.Path.cwd().parents[1]
_hspec = importlib.util.spec_from_file_location("_harness", ROOT / "tests" / "_harness.py")
h = importlib.util.module_from_spec(_hspec); _hspec.loader.exec_module(h)

### Parameters

In [ ]:
# ---- design knobs: choose your own numbers ----
P, I          = 1500, 30
K             = 50          # simulated datasets per scenario (the main cost driver)
N_ITER_PROBIT = 4000        # probit = Albert-Chib Gibbs (mixes fast, cheaper)
N_ITER_LOGIT  = 8000        # logit  = Metropolis-Hastings (give it more)
N_CHAINS      = 2
SEED          = 20250413

# Truth is FIXED per scenario; only the response draw changes across the K
# datasets ("fix everything, redraw only responses").  Evidence-mode warmup is
# the package default, so each fit adapts its own warmup.

### Helpers

Fixed-truth Monte-Carlo machinery, shared by every scenario.

In [ ]:
def responses_frame(Y):
    """[P, I] response matrix -> keyed wide table for ingest.load_data."""
    Y = np.asarray(Y); Pn, In = Y.shape
    df = pd.DataFrame(Y, columns=[f'item{i+1}' for i in range(In)])
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(Pn)])
    return df

def person_frame(X, names):
    X = np.asarray(X)
    df = pd.DataFrame(X, columns=list(names))
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(X.shape[0])])
    return df

def item_frame(Q, names):
    Q = np.asarray(Q)
    df = pd.DataFrame(Q, columns=list(names))
    df.insert(0, 'item_id', [f'item{i+1}' for i in range(Q.shape[0])])
    return df


class MCStore:
    """Collects per-dataset recovery for a set of parameter-name PREFIXES.
    Truth is fixed; each dataset is one fresh response realization."""
    def __init__(self, truth_map):
        self.truth = {p: np.asarray(v, float).ravel() for p, v in truth_map.items()}
        self.post  = {p: [] for p in truth_map}
        self.lo    = {p: [] for p in truth_map}
        self.hi    = {p: [] for p in truth_map}
        self.tables   = []         # per-dataset DataFrame (all prefixes), kept for drill-down
        self.last_fit = None

    def add(self, fit):
        self.last_fit = fit
        names = fit.results[0].param_names
        draws = np.concatenate([np.asarray(r.chain, float) for r in fit.results])
        rowtabs = []
        for p, tv in self.truth.items():
            cols = [j for j, n in enumerate(names) if n.startswith(p)]
            if not cols:
                raise KeyError(f"no chain columns start with {p!r}; "
                               f"first names are {names[:8]}")
            d  = draws[:, cols]
            pm = d.mean(0); lo = np.percentile(d, 2.5, 0); hi = np.percentile(d, 97.5, 0)
            m  = min(len(tv), len(cols))
            self.post[p].append(pm[:m]); self.lo[p].append(lo[:m]); self.hi[p].append(hi[:m])
            rowtabs.append(pd.DataFrame({'param': [names[j] for j in cols][:m],
                                         'true': tv[:m], 'post_mean': pm[:m],
                                         'ci_lo': lo[:m], 'ci_hi': hi[:m]}))
        self.tables.append(pd.concat(rowtabs, ignore_index=True))

    def summary(self):
        rows = []
        for p, tv in self.truth.items():
            Pm = np.array(self.post[p]); Lo = np.array(self.lo[p]); Hi = np.array(self.hi[p])  # [K, n]
            m  = min(tv.shape[0], Pm.shape[1]); t = tv[:m]
            Pm, Lo, Hi = Pm[:, :m], Lo[:, :m], Hi[:, :m]
            err   = Pm - t[None, :]                                       # [K, n]
            cover = ((Lo <= t[None, :]) & (t[None, :] <= Hi)).mean(0)     # per element, over K
            rows.append(dict(param=p, n=m, K=Pm.shape[0],
                             mean_bias=float(err.mean()),
                             mean_abs_bias=float(np.abs(err.mean(0)).mean()),
                             rmse=float(np.sqrt((err ** 2).mean())),
                             coverage95=float(cover.mean())))
        return pd.DataFrame(rows)


def run_recovery(K, spec, gen_Y, make_data, make_models, n_iter, truth_map,
                 seed0, n_chains=None, make_priors=None):
    """Fix the truth; draw K response realizations; fit each; collect recovery."""
    n_chains = N_CHAINS if n_chains is None else n_chains
    store = MCStore(truth_map)
    for k in range(K):
        Y    = gen_Y(np.random.default_rng(seed0 + 1000 + k))   # response realization k
        data = make_data(Y)
        mods = make_models()
        priors = make_priors(mods) if make_priors is not None else None
        fit  = ft.irtree_fit(data, spec, mods, priors=priors,
                             n_iter=n_iter, n_chains=n_chains, seed=seed0 + k)
        store.add(fit)
    return store


def show_recovery(store, title):
    """Print the aggregate coverage/bias/RMSE table for one link."""
    print(title)
    s = store.summary()
    fmt = {c: '{:+.4f}'.format for c in ['mean_bias', 'mean_abs_bias', 'rmse']}
    fmt['coverage95'] = '{:.3f}'.format
    print(s.to_string(index=False, formatters=fmt))
    print()
    return s


def dataset_table(store, k):
    """Per-dataset posterior-mean-vs-truth table for dataset k (drill-down)."""
    t = store.tables[k].copy()
    t['abs_err'] = (t['post_mean'] - t['true']).abs()
    print(f'dataset {k}:')
    fmt = {c: '{:+.3f}'.format for c in ['true', 'post_mean', 'ci_lo', 'ci_hi', 'abs_err']}
    print(t.to_string(index=False, formatters=fmt))
    return t

### How to read this notebook

Each scenario fits **K independently simulated datasets** that share one fixed truth (only the response draw changes). Probit and logit are separate cells so you can give each its own iteration count (`N_ITER_PROBIT`, `N_ITER_LOGIT`).

Each link prints one aggregate table: per parameter group, `mean_bias`, `mean_abs_bias`, `rmse`, and `coverage95` (the fraction of the K fits whose 95% interval contains the truth; well-calibrated ≈ 0.95). The per-dataset tables are retained — call `dataset_table(store, k)` on any scenario’s store (e.g. `A1p`, `A1l`) to see dataset k in full.

**Reading the identifiability probes.** `κ` is identified through b's prior: κ is the regression of the item shifts on `Q`, with the `b_i` as residuals. With `b` free (the default hierarchical prior) the package's sampler, which interweaves κ and `b`, should recover κ; a tight `b` prior at 0 narrows the κ intervals. The tight prior is valid here because the probe sets true `b = 0`, so all item-axis signal comes from `Q·κ`. (Notebook 30 shows why a plain two-block Gibbs sampler appeared to "collapse" κ: it stays wherever it starts.)

## 1. `κ` / `b` — identified through b's prior (item axis)

`b_i` and `Q_i·κ` are both per-item shifts. Without a prior on `b` only their sum would be identified; with `b_i ~ N(0, τ²_b)` κ is the regression of the item shifts on `Q`. With true `b = 0` the hierarchical prior learns a small `τ²_b`, so κ should be recovered with `b` free, and more tightly when `b` is pinned at 0 (its truth).

**Simulate** (fixed truth; only responses redrawn across K)

In [ ]:
# Probe 1: kappa / b on the item axis. True b = 0, so item-axis signal is all Q.kappa.
spec_cur = ts.disengagement_tree(binary=True); seed_scn = SEED + 30
rng = np.random.default_rng(seed_scn)
Qmat = rng.normal(size=(I,2)); kappa_true = np.array([0.5, -0.4])
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1 = np.zeros(I); b2 = rng.normal(0,1,I)            # node-1 difficulty truly 0
off1 = (Qmat @ kappa_true)[None, :].repeat(P, 0)
truth = [dict(theta=theta1, b=b1, offset=off1), dict(theta=theta2, b=b2)]
truth_map = {'kappa_1_': kappa_true}
qdf = item_frame(Qmat, ['q0','q1'])
def gen_pro(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['probit','probit'])
def gen_log(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['logit','logit'])
def load_cur(Y): return ing.load_data(responses_frame(Y), spec=spec_cur, item_covariates=qdf, categories_start=1)
make_pro = lambda: [nm.ProbitRaschNode(item_covariates=['q0','q1']), nm.ProbitRaschNode()]
make_log = lambda: [nm.LogitRaschNode(item_covariates=['q0','q1']),  nm.LogitRaschNode()]
def mk_tight_b(mods):
    p = pr.build_default_priors(spec_cur, mods)
    p.set('b', pr.NormalPrior(mean=0.0, variance=0.01))   # pin b ~ 0 == truth
    return p

### 1a. Free `b` (default hierarchical prior) → `κ` recovered

**Probit**

In [ ]:
K1 = min(K, 20)   # identifiability probes are cheap to read; dial if you like
P1a_p = run_recovery(K1, spec_cur, gen_pro, load_cur, make_pro, N_ITER_PROBIT, truth_map, seed_scn)
show_recovery(P1a_p, '1a PROBIT  free b -> kappa recovered (identified through b prior)')

**Logit**

In [ ]:
P1a_l = run_recovery(K1, spec_cur, gen_log, load_cur, make_log, N_ITER_LOGIT, truth_map, seed_scn)
show_recovery(P1a_l, '1a LOGIT  free b -> kappa recovered (identified through b prior)')

### 1b. Tight `b` prior (= truth 0) → `κ` recovered with narrower intervals

**Probit**

In [ ]:
P1b_p = run_recovery(K1, spec_cur, gen_pro, load_cur, make_pro, N_ITER_PROBIT, truth_map, seed_scn, make_priors=mk_tight_b)
show_recovery(P1b_p, '1b PROBIT  tight b (== truth 0) -> kappa should RECOVER')

**Logit**

In [ ]:
P1b_l = run_recovery(K1, spec_cur, gen_log, load_cur, make_log, N_ITER_LOGIT, truth_map, seed_scn, make_priors=mk_tight_b)
show_recovery(P1b_l, '1b LOGIT  tight b (== truth 0) -> kappa should RECOVER')

### 1c. Sweep the `b` prior → κ's precision tracks the constraint

Heaviest cell (grid × both links × `Ksw` datasets); it reruns the fit per grid point. Reduce `Ksw` for a quick look. Expect coverage near 0.95 throughout and RMSE shrinking as the `b` prior tightens around its truth (0): the tighter the prior, the less of the item shift `b` can take, and the more precisely κ is pinned down.


In [ ]:
# 1c: sweep the b-prior variance; kappa's precision should track the constraint.
def mk_bvar(v):
    def f(mods):
        p = pr.build_default_priors(spec_cur, mods)
        p.set('b', pr.NormalPrior(mean=0.0, variance=v))
        return p
    return f
Ksw = min(K, 8)
for link, gen, mk in [('probit', gen_pro, make_pro), ('logit', gen_log, make_log)]:
    nit = N_ITER_PROBIT if link == 'probit' else N_ITER_LOGIT
    print(f'b-prior sweep ({link}):')
    for v in [4.0, 1.0, 0.25, 0.05, 0.01]:
        st = run_recovery(Ksw, spec_cur, gen, load_cur, mk, nit, truth_map, seed_scn, make_priors=mk_bvar(v))
        s = st.summary().iloc[0]
        print(f'    tau2_b={v:5.2f}  kappa coverage={s.coverage95:.2f}  rmse={s.rmse:+.3f}')

## 2. `β` / `θ` — the identified pair (different axes)

**Simulate** (fixed truth; only responses redrawn across K)

In [ ]:
# Probe 2: beta / theta are identified (different axes). Centred X, proper theta prior.
spec_cur = ts.disengagement_tree(binary=True); seed_scn = SEED + 31
rng = np.random.default_rng(seed_scn)
Xmat = rng.normal(size=(P,2)); Xmat -= Xmat.mean(0)
beta_true = np.array([0.6, -0.5])
theta1 = rng.normal(0,1,P) + Xmat @ beta_true; theta2 = rng.normal(0,1,P)
b1 = rng.normal(0,0.7,I); b2 = rng.normal(0,1,I)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2)]
truth_map = {'beta_1_': beta_true}
xdf = person_frame(Xmat, ['x0','x1'])
def gen_pro(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['probit','probit'])
def gen_log(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['logit','logit'])
def load_cur(Y): return ing.load_data(responses_frame(Y), spec=spec_cur, person_covariates=xdf, categories_start=1)
make_pro = lambda: [nm.ProbitRaschNode(person_covariates=['x0','x1']), nm.ProbitRaschNode()]
make_log = lambda: [nm.LogitRaschNode(person_covariates=['x0','x1']),  nm.LogitRaschNode()]

**Probit** — K datasets, `N_ITER_PROBIT`

In [ ]:
P2_p = run_recovery(min(K,20), spec_cur, gen_pro, load_cur, make_pro, N_ITER_PROBIT, truth_map, seed_scn)
show_recovery(P2_p, '2 PROBIT  beta identified (coverage ~ 0.95 expected)')

**Logit** — K datasets, `N_ITER_LOGIT`

In [ ]:
P2_l = run_recovery(min(K,20), spec_cur, gen_log, load_cur, make_log, N_ITER_LOGIT, truth_map, seed_scn)
show_recovery(P2_l, '2 LOGIT  beta identified (coverage ~ 0.95 expected)')

`β` multiplies a fixed observed `X_p`; `θ_p` is a free per-person effect on the same latent, but with centred `X` and a proper `θ` prior they are identified together — so `β` recovers with free `θ` under both links.

## Summary

The probes assert the *right* behaviour per pair: `κ` is identified through b's prior (weakly when the prior is wide, since the information comes from the I item locations) and the package's sampler recovers it with `b` free, more tightly as `b` is constrained; `β`/`θ` are identified on different axes (β recovers either way). Run under both links, the result is a statement about the model, not the link or the draw.
